In [ ]:
### UMAP visualizations

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
import umap
from sklearn.metrics import silhouette_score

In [ ]:
# Read CSVs
df_intensity_norm = pd.read_csv("dataframe_w_intensities_normalized.csv", delimiter=",", quotechar='|', index_col = None)

df_intensity_norm = df_intensity_norm[
    [c for c in df_intensity_norm.columns if c.startswith('mean') and c.endswith('normalized')]
]
print(df_intensity_norm)

In [14]:
numeric_df = df_intensity_norm.select_dtypes(include=[np.number])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(numeric_df.values)

In [ ]:
import random
#### KMEANS
scores = {}
for i in range(2,20):
    kmeans = MiniBatchKMeans(
        n_clusters=i,
        batch_size=16384,
        max_iter=200,
        n_init=3,
        random_state=42
    )
    labels = kmeans.fit_predict(X_scaled)
    centroids = kmeans.cluster_centers_ 
    
    score = silhouette_score(
        X_scaled,
        labels,
        sample_size=5000,   # even 5000 often fine
        random_state=42
    )
    print(
    "For n_clusters =",
    f'{i}',
    "The average silhouette_score is :",
    score,
    )

In [5]:
from sklearn.utils import shuffle

save_dir = './images/normalized'
                            
reducer= umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    densmap=False,
    metric="euclidean",
    random_state=42,
    low_memory = True,
    n_jobs = 8
)

X_scaled = X_scaled.astype(np.float32)
embedding = reducer.fit_transform(X_scaled)

In [32]:
embedding = pd.read_csv("UMAP_Embedding.csv")

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(embedding['0'], embedding['1'], c=labels, cmap='tab10', s=5)
plt.scatter(embedding['0'], embedding['1'], c=labels, cmap='tab10', s=5, rasterized=True)
plt.savefig("umap_norm_new.svg", dpi=100)
plt.colorbar(label='KMeans cluster')
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
plt.title('UMAP embedding colored by KMeans clusters')
plt.gca().set_aspect('equal', 'datalim')
save_name = "umap_norm_new.svg"
plt.savefig(save_name, dpi=100)
plt.show()

In [8]:
## Save the coordinates as a 
df_c = pd.DataFrame(embedding)
df_c.to_csv("UMAP_Embedding.csv")